# Onion Quality Assessment — End-to-End Colab Training & Local Export
**Handheld Onion Quality Assessment & Multi-Standard Grading (DoCA + AGMARK)**

This workbook runs in **Google Colab via the VS Code extension**.
It executes the complete Phase 2 model training pipeline:
- **Model 1**: YOLOv8-seg single-class `onion` detector & segmenter (Target: mAP@0.5 ≥ 85% on D1-B benchmark)
- **Model 2**: 4-class defect CNN on bulb crops (`healthy`, `sprouted`, `rotten`, `mechanical_damage`) (Target: F1 ≥ 90%)
- **Export**: Lightweight ONNX models for handheld mobile inference
- **Local Download**: Programmatic transfer cell allowing instant download to your local PC without using Colab's web UI.

### Step 1: Environment & GPU Verification

In [4]:
!nvidia-smi
!pip install -q "ultralytics>=8.3.0" "roboflow>=1.1.0" "opencv-python-headless>=4.10.0" "pydantic>=2.8.0" "imagehash>=4.3.1" pandas matplotlib tqdm pyyaml

Sun Sep 20 12:29:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Download Public Datasets via Roboflow API
Enter your free Roboflow API key below to download D1-A, D1-B, and D2-A directly onto Colab's high-speed cloud disk.

In [5]:
import os
from roboflow import Roboflow

# Enter your Roboflow Private API Key here:
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "Iro3JVh4TIt3iqgCNxpl")

os.makedirs("/content/data/raw", exist_ok=True)
os.makedirs("/content/runs", exist_ok=True)
os.makedirs("/content/exports", exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

print("1. Downloading D1-A (YOLOv8 instance segmentation)... ")
d1a_project = rf.workspace("yolo-custom-object-detection").project("instance-segmentation-wagk9")
d1a_data = d1a_project.version(12).download("yolov8", location="/content/data/raw/D1A_instance_seg")

print("\n2. Downloading D1-B (onionthesis1 - cross-source test set)... ")
d1b_project = rf.workspace("paul-angelo-lavarias-ii").project("onionthesis1")
d1b_data = d1b_project.version(1).download("yolov8", location="/content/data/raw/D1B_onionthesis1")

print("\n3. Downloading D2-A (onion-disease - defect classifier source)... ")
d2a_project = rf.workspace("anas-kadiri-ffnxr").project("onion-disease")
d2a_data = d2a_project.version(1).download("yolov8", location="/content/data/raw/D2A_onion_disease")

print("\n All datasets downloaded successfully into /content/data/raw/")

1. Downloading D1-A (YOLOv8 instance segmentation)... 
loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to /content/data/raw/D1A_instance_seg in yolov8:: 100%|██████████| 6957/6957 [00:04<00:00, 1436.68it/s]



2. Downloading D1-B (onionthesis1 - cross-source test set)... 
loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to /content/data/raw/D1B_onionthesis1 in yolov8:: 100%|██████████| 15053/15053 [00:02<00:00, 5710.12it/s]



3. Downloading D2-A (onion-disease - defect classifier source)... 
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/data/raw/D2A_onion_disease in yolov8:: 100%|██████████| 262/262 [00:00<00:00, 9413.86it/s]


 All datasets downloaded successfully into /content/data/raw/


### Step 3: Prepare Model 1 Dataset (Single-Class 'onion')
Unifies all masks to class `0: onion`, cleans negative labels from D1-B, and generates `data.yaml`.

In [12]:
import shutil
from pathlib import Path
import yaml

def get_onion_class_indices(raw_dir: str) -> list[int]:
    yaml_path = Path(raw_dir) / "data.yaml"
    if not yaml_path.exists():
        return [0]
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    names = cfg.get("names", [])
    print(f"Classes found in {Path(raw_dir).name}: {names}")
    if isinstance(names, dict):
        matches = [int(idx) for idx, name in names.items() if "not" not in str(name).lower() and "onion" in str(name).lower()]
        return matches if matches else [0]
    elif isinstance(names, list):
        matches = [idx for idx, name in enumerate(names) if "not" not in str(name).lower() and "onion" in str(name).lower()]
        return matches if matches else [0]
    return [0]

def prepare_model1_split(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, target_classes=[0]):
    Path(dst_img_dir).mkdir(parents=True, exist_ok=True)
    Path(dst_lbl_dir).mkdir(parents=True, exist_ok=True)
    
    count = 0
    for img_p in Path(src_img_dir).glob("*.jpg"):
        lbl_p = Path(src_lbl_dir) / f"{img_p.stem}.txt"
        dest_img = Path(dst_img_dir) / img_p.name
        dest_lbl = Path(dst_lbl_dir) / f"{img_p.stem}.txt"
        
        shutil.copy2(img_p, dest_img)
        if not lbl_p.exists():
            dest_lbl.touch()
            continue
            
        with open(lbl_p, "r") as f:
            lines = f.readlines()
        valid = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 6 and int(parts[0]) in target_classes:
                valid.append("0 " + " ".join(parts[1:]) + "\n")
        with open(dest_lbl, "w") as f:
            f.writelines(valid)
        count += 1
    return count

d1a_classes = get_onion_class_indices("/content/data/raw/D1A_instance_seg")
d1b_classes = get_onion_class_indices("/content/data/raw/D1B_onionthesis1")
print(f"\nTarget onion class IDs -> D1-A: {d1a_classes}, D1-B: {d1b_classes}")

# Check D1-B available split folders
d1b_root = Path("/content/data/raw/D1B_onionthesis1")
d1b_test_img = d1b_root / "test" / "images"
d1b_test_lbl = d1b_root / "test" / "labels"
if not d1b_test_img.exists():
    d1b_test_img = d1b_root / "valid" / "images"
    d1b_test_lbl = d1b_root / "valid" / "labels"
if not d1b_test_img.exists():
    d1b_test_img = d1b_root / "train" / "images"
    d1b_test_lbl = d1b_root / "train" / "labels"

print("Formatting Model 1 dataset with dynamic class resolution...")
prepare_model1_split("/content/data/raw/D1A_instance_seg/train/images", "/content/data/raw/D1A_instance_seg/train/labels", "/content/data/model1/train/images", "/content/data/model1/train/labels", target_classes=d1a_classes)
prepare_model1_split("/content/data/raw/D1A_instance_seg/valid/images", "/content/data/raw/D1A_instance_seg/valid/labels", "/content/data/model1/val/images", "/content/data/model1/val/labels", target_classes=d1a_classes)
prepare_model1_split(d1b_test_img, d1b_test_lbl, "/content/data/model1/test/images", "/content/data/model1/test/labels", target_classes=d1b_classes)

data_yaml = {
    "path": "/content/data/model1",
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {0: "onion"}
}
with open("/content/data/model1/data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

print("\n Model 1 data.yaml ready at /content/data/model1/data.yaml")

Classes found in D1A_instance_seg: ['Onion']
Classes found in D1B_onionthesis1: ['Not Onion', 'Onion']

Target onion class IDs -> D1-A: [0], D1-B: [1]
Formatting Model 1 dataset with dynamic class resolution...

 Model 1 data.yaml ready at /content/data/model1/data.yaml


### Step 4: Train Model 1 (YOLOv8n-seg)
Target: **mAP@0.5 ≥ 85%** on cross-source benchmark.

In [8]:
from ultralytics import YOLO

# Load pretrained YOLOv8-nano segmentation weights
model1 = YOLO("yolov8n-seg.pt")

# Fast Fine-Tuning with RAM Caching & Early Stopping
results_m1 = model1.train(
    data="/content/data/model1/data.yaml",
    epochs=30,        # Max epochs (stops earlier if converged)
    patience=5,       # Early stopping: stop if validation mAP doesn't improve for 5 epochs
    imgsz=640,        # Set to 512 for even faster ~12s epochs
    batch=32,         # 2x batch size: doubles GPU throughput on T4
    cache="ram",      # Load images into RAM: 3x-5x faster (no repeated disk reads)
    workers=2,        # Matches Colab 2 vCPU limit (eliminates CPU bottlenecks)
    close_mosaic=5,   # Disables mosaic for final 5 epochs for fast clean convergence
    degrees=10.0,
    fliplr=0.5,
    flipud=0.5,
    mosaic=0.5,
    project="/content/runs/model1",
    name="yolov8_seg_onion",
    save=True,
    plots=True
)

Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/model1/data.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolov8_seg_onion-3, nbs=64, nms=None, opset=

### Step 5: Evaluate Model 1 on D1-B Cross-Source Benchmark
Tests the model on an independent author's dataset (unseen background and lighting).

In [13]:
# 1. Benchmark on D1-B test split
print("Evaluating on D1-B cross-source test set...")
metrics_m1 = model1.val(data="/content/data/model1/data.yaml", split="test")
print("=" * 60)
print(f"Model 1 D1-B Benchmark Box mAP@0.5:  {metrics_m1.box.map50:.4f}")
print(f"Model 1 D1-B Benchmark Mask mAP@0.5: {metrics_m1.seg.map50:.4f}")
print("=" * 60)

# 2. Compare against D1-A held-out test split (if present)
if Path("/content/data/raw/D1A_instance_seg/data.yaml").exists():
    try:
        metrics_d1a = model1.val(data="/content/data/raw/D1A_instance_seg/data.yaml", split="test")
        print(f"Model 1 on D1-A Test Split Mask mAP@0.5: {metrics_d1a.seg.map50:.4f}")
        print("=" * 60)
    except Exception as e:
        print(f"D1-A test split check note: {e}")

# 3. Visual Inspection: Predict and overlay masks on 3 sample D1-B test images
test_images = list(Path("/content/data/model1/test/images").glob("*.jpg"))[:3]
if test_images:
    preds = model1.predict(source=test_images, save=True, project="/content/runs/model1", name="d1b_sample_preds", conf=0.25)
    print(f"Visual sample overlays saved to: /content/runs/model1/d1b_sample_preds/")

Evaluating on D1-B cross-source test set...
Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2125.4±663.9 MB/s, size: 72.5 KB)
val: Scanning /content/data/model1/test/labels... 6896 images, 4656 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6896/6896 1.7Kit/s 4.0s0.1s
val: /content/data/model1/test/images/IMG_2023-04-22-16-30-10-121_jpg.rf.32e7c4d4cb62ede286f4f47138ac780d.jpg: 1 duplicate labels removed
val: New cache created: /content/data/model1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 431/431 5.9it/s 1:130.1sss
                   all       6896       8076       0.23      0.702      0.221      0.179       0.23      0.705       0.22      0.176
Speed: 1.2ms preprocess, 5.0ms inference, 0.0ms lo

### Step 6: Prepare Model 2 Defect Crops (4 Core Classes)
Extracts individual bulb crops from D2-A and categorizes them into `healthy`, `sprouted`, `rotten`, `mechanical_damage`.

In [ ]:
import cv2
from pathlib import Path
import random
import yaml

def find_label_file(img_path: Path) -> Path | None:
    # 1. Same directory
    lbl = img_path.with_suffix(".txt")
    if lbl.exists(): return lbl
    # 2. Path replacing /images/ with /labels/
    parts = list(img_path.parts)
    if "images" in parts:
        idx = len(parts) - 1 - parts[::-1].index("images")
        parts[idx] = "labels"
        lbl = Path(*parts).with_suffix(".txt")
        if lbl.exists(): return lbl
    # 3. Sibling labels folder
    lbl = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
    if lbl.exists(): return lbl
    return None

def map_defect_class(raw_name: str) -> str | None:
    name = str(raw_name).strip().lower()
    if "rot" in name: return "rotten"
    if "sprout" in name or "root" in name: return "sprouted"
    if "cut" in name or "damage" in name: return "mechanical_damage"
    if "healthy" in name or name == "onion": return "healthy"
    return None

classes = ["healthy", "sprouted", "rotten", "mechanical_damage"]
for s in ["train", "val", "test"]:
    for c in classes:
        Path(f"/content/data/model2/{s}/{c}").mkdir(parents=True, exist_ok=True)

# Read data.yaml from D2A
yaml_p = Path("/content/data/raw/D2A_onion_disease/data.yaml")
raw_names = []
if yaml_p.exists():
    with open(yaml_p) as f:
        d2a_cfg = yaml.safe_load(f)
    raw_names = d2a_cfg.get("names", [])
    if isinstance(raw_names, dict):
        raw_names = [raw_names[k] for k in sorted(raw_names.keys())]

print(f"D2-A Class Names from data.yaml: {raw_names}")

crops_by_class = {c: [] for c in classes}
img_extensions = {".jpg", ".jpeg", ".png"}
images = [p for p in Path("/content/data/raw/D2A_onion_disease").rglob("*") if p.suffix.lower() in img_extensions]
print(f"Found {len(images)} images in D2-A. Extracting crops...")

for img_p in images:
    lbl_p = find_label_file(img_p)
    if not lbl_p or not lbl_p.exists(): continue
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    
    with open(lbl_p) as f: lines = f.readlines()
    for idx, l in enumerate(lines):
        p = l.strip().split()
        if len(p) < 5: continue
        cid = int(p[0])
        cname = raw_names[cid] if raw_names and cid < len(raw_names) else str(cid)
        target_cls = map_defect_class(cname)
        if not target_cls: continue
        
        cx, cy, bw, bh = float(p[1]), float(p[2]), float(p[3]), float(p[4])
        x1, y1 = max(0, int((cx - bw*0.54)*w)), max(0, int((cy - bh*0.54)*h))
        x2, y2 = min(w, int((cx + bw*0.54)*w)), min(h, int((cy + bh*0.54)*h))
        if x2 > x1 and y2 > y1:
            crop = cv2.resize(img[y1:y2, x1:x2], (224, 224))
            crops_by_class[target_cls].append((crop, f"{img_p.stem}_c{idx}.jpg"))

# Save train (70%), val (15%), test (15%)
for c, items in crops_by_class.items():
    random.shuffle(items)
    n1 = int(len(items)*0.70)
    n2 = int(len(items)*0.85)
    for crop, name in items[:n1]: cv2.imwrite(f"/content/data/model2/train/{c}/{name}", crop)
    for crop, name in items[n1:n2]: cv2.imwrite(f"/content/data/model2/val/{c}/{name}", crop)
    for crop, name in items[n2:]: cv2.imwrite(f"/content/data/model2/test/{c}/{name}", crop)
    print(f"  - {c}: {len(items)} crops")

print("\n Model 2 ImageFolder dataset ready at /content/data/model2")

### Step 7: Train Model 2 (Defect Classifier)
Target: **F1 ≥ 90%** on defect classes.

In [ ]:
model2 = YOLO("yolov8n-cls.pt")

# Fast Classifier Training with RAM Caching & Early Stopping
results_m2 = model2.train(
    data="/content/data/model2",
    epochs=25,        # Max epochs
    patience=5,       # Early stopping: stop if validation metric plateaus
    imgsz=224,        # Standard crop input size
    batch=64,         # Fast batch size on T4 for 224x224 crops
    cache="ram",      # Load all crops into RAM
    workers=2,        # Matches Colab 2 vCPUs
    project="/content/runs/model2",
    name="defect_classifier_4class",
    save=True,
    plots=True
)

metrics_m2 = model2.val(data="/content/data/model2", split="test")
print(f"\nModel 2 Top-1 Test Accuracy: {metrics_m2.top1:.4f}")

### Step 8: Export Both Models to ONNX for Mobile Deployment

In [ ]:
# Export Model 1 (detection + segmentation) with dynamic shape for mobile resolution flexibility
onnx_m1 = model1.export(format="onnx", dynamic=True)
# Export Model 2 (defect classifier) at fixed 224x224
onnx_m2 = model2.export(format="onnx", imgsz=224)

print(f"\n Exported Model 1 ONNX: {onnx_m1}")
print(f" Exported Model 2 ONNX: {onnx_m2}")

### Step 9: Download Trained Models Directly to Your PC (No Colab Web UI Needed)
Run the cell below. It packages all `.pt` weights, `.onnx` models, and evaluation plots, uploads them, and gives you an exact 1-line `curl` command to run in your local PC terminal.

In [ ]:
import subprocess

# 1. Organize exports
!mkdir -p /content/exports/weights /content/exports/onnx /content/exports/eval_plots
!cp /content/runs/model1/yolov8_seg_onion/weights/best.pt /content/exports/weights/model1_yolov8_seg.pt || true
!cp /content/runs/model2/defect_classifier_4class/weights/best.pt /content/exports/weights/model2_defect_cls.pt || true
!cp /content/runs/model1/yolov8_seg_onion/weights/best.onnx /content/exports/onnx/model1_yolov8_seg.onnx || true
!cp /content/runs/model2/defect_classifier_4class/weights/best.onnx /content/exports/onnx/model2_defect_cls.onnx || true
!cp /content/runs/model1/yolov8_seg_onion/*.png /content/exports/eval_plots/ || true
!cp /content/runs/model2/defect_classifier_4class/*.png /content/exports/eval_plots/ || true

# 2. Zip archive
archive_path = "/content/onion_model_artifacts.zip"
!zip -r {archive_path} /content/exports

# 3. Programmatic CLI Download via transfer.sh
print("\nUploading package to transfer.sh for direct terminal download...")
res = subprocess.run(f"curl --upload-file {archive_path} https://transfer.sh/onion_model_artifacts.zip", shell=True, capture_output=True, text=True)
url = res.stdout.strip()

print("\n" + "="*80)
print(" TRAINING COMPLETE! DOWNLOAD ARTIFACTS TO YOUR PC WITH THIS COMMAND:")
print("="*80)
print(f"\nmkdir -p ~/Projects/Onion_grading_new/runs && curl -o ~/Projects/Onion_grading_new/runs/onion_model_artifacts.zip {url} && unzip -o ~/Projects/Onion_grading_new/runs/onion_model_artifacts.zip -d ~/Projects/Onion_grading_new/runs/\n")
print("="*80)

### (Optional Alternative) Google Drive Mount Backup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/onion_grading_runs
    !cp /content/onion_model_artifacts.zip /content/drive/MyDrive/onion_grading_runs/
    print("\n Saved backup to Google Drive at: /content/drive/MyDrive/onion_grading_runs/")
except Exception as e:
    print(f"Drive backup skipped: {e}")